Notebook that aggregate Gamma HMC and Neutra Chains together, and perform diagnostics

In [1]:
import time
from codebase_0 import CANN, MLP, cann_training_loop, mlp_train_loop, DeviceLoader, set_seed
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from torch import nn
import torch
import pyro

from gamma_bcann_vi import (
    gamma_build_bcann_guide, GammaBCANN, gamma_fit_glm_mle,
    gamma_bcann_svi_training_loop, GammaCANN, gamma_cann_training_loop,
    gamma_deviance, gamma_evaluate_bcann
)

from gamma_bcann_hmc_nuts import (
    gamma_nuts_diagnostics, gamma_predict_from_samples, gamma_nuts_posterior_stats, gamma_deviance_from_stats, gamma_svi_posterior_stats
)

from bcann_neutra import (
    neutra_diagnostics, neutra_posterior_stats, neutra_deviance_from_stats
)

from gamma_horseshoe import (
    HorseshoeBayesianCANN, HorseshoeGammaBCANN
)

seed = 42
set_seed(seed)
pyro.clear_param_store()

/Users/thangmai/Downloads/UNIMELB/HONOURS RESEARCH (ACTL40001)/.venv-honours/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load data and train/validation/test split

In [2]:
device = torch.device("cpu")

torch.set_default_device("cpu")

dataset = pd.read_csv("freMTPL2sev_transformed.csv", index_col= 0) 
y = torch.tensor(dataset["ClaimAmount"].to_numpy(), dtype=torch.float32)
weights = torch.ones(len(y)) 
X = torch.tensor(
    dataset.drop(columns=["IDpol", "Exposure","ClaimAmount"]).to_numpy(), 
    dtype=torch.float32
)   # 48 columns

# Learning-test set split
X_learn, X_test, y_learn, y_test, w_learn, w_test = train_test_split(
    X, y, weights, test_size= 0.1, random_state= seed
)

# Training-CV set split
X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X_learn, y_learn, w_learn, test_size= 0.1, random_state= seed
)

learn_scaler = MinMaxScaler().fit(X_learn)
X_learn = learn_scaler.transform(X_learn)
X_test = learn_scaler.transform(X_test)

train_scaler = MinMaxScaler().fit(X_train)
X_train = train_scaler.transform(X_train)
X_val = train_scaler.transform(X_val)

def T(a):
    return torch.as_tensor(a, dtype=torch.float32, device=device)

X_train, X_val, X_learn, X_test = map(T, (X_train, X_val, X_learn, X_test))
y_train, y_val, y_learn, y_test = map(T, (y_train, y_val, y_learn, y_test))
w_train, w_val, w_learn, w_test = map(T, (w_train, w_val, w_learn, w_test))

torch.set_default_device(device)
train_loader = DeviceLoader(
        X_train, y_train, w_train,
        batch_size = 2056,
        shuffle= True
)
val_loader = DeviceLoader(
        X_val, y_val, w_val,
        batch_size= 10000
)
learn_loader = DeviceLoader(
        X_learn, y_learn, w_learn,
        batch_size= 2056
)
test_loader = DeviceLoader(
        X_test, y_test, w_test,
        batch_size= 10000
)

Start a GammaBCANN model

In [35]:
network_architecture = [20,15, 10]
gamma_bcann = GammaBCANN(
    in_features= 48,
    neural_net_params={"hidden_activation_nodes": network_architecture,
                        "hidden_activations": nn.Tanh}
).to(device)

# NUTS diagnostics

Load NUTS Samples from disks and run diagnostics

In [ ]:
nuts_run = torch.load("hmc_results/gamma_hmcnuts_full_set.pt", map_location="cpu")

nuts_samples, nuts_info = nuts_run["samples"], nuts_run["info"]

In [ ]:
nuts_rows = gamma_nuts_diagnostics(
    model=gamma_bcann, samples=nuts_samples, info=nuts_info,
    X_eval=X_learn, weights_eval=w_learn, y_eval=y_learn,
    n_points=500, seed=seed, device=device,
)

Device used             : cuda:2
Wall time per chain (s) : [2430.2, 2452.4, 1874.7, 1907.0, 2457.2, 1912.6]
Divergences per chain   : [0, 0, 0, 0, 0, 0]
Adapted step size       : [0.04752, 0.0464, 0.05457, 0.05276, 0.04609, 0.05758]
Gamma shape (posterior) : mean 0.784, 95% CI [0.772, 0.797]
quantity                                 max R-hat   min ESS
mu                                           1.001      5790
eta_glm                                      1.002      3949
f_nn                                         1.000      3904
shape                                        0.999     10054
cann.linear_pred.layer.bias                  1.000      5124
cann.linear_pred.layer.weight                1.001      6765
loglik (eval subset)                         1.000      6512


Making predictions

In [41]:
nuts_stats = gamma_nuts_posterior_stats(gamma_bcann, nuts_samples, X_test, batch = 500)

In [42]:
print(gamma_deviance_from_stats(nuts_stats, y_test))

1.8306804895401


# NeuTra diagnostics

Load NeuTra Samples from disks and run diagnostics

In [ ]:
neutra_run = torch.load("hmc_results/gamma_neutra_full_set.pt", map_location="cpu")

neutra_samples, neutra_info = neutra_run["samples"], neutra_run["info"] 

In [ ]:
neutra_rows = neutra_diagnostics(
    model=gamma_bcann, samples=neutra_samples, info=neutra_info,
    X_eval=X_learn, aux_eval= w_learn, y_eval=y_learn,
    n_points=500, seed=seed, device=device,
)

Device used             : ['cuda:3', 'cuda:3', 'cuda:3', 'cuda:3', 'cuda:3', 'cuda:3']
Wall time per chain (s) : [3025.4, 3003.0, 3031.0, 2987.0, 3063.3, 3122.5]
Divergences per chain   : [0, 0, 0, 0, 0, 0]
Adapted step size       : [0.0383, 0.03821, 0.03295, 0.0399, 0.03862, 0.03576]
Gamma shape (posterior) : mean 0.784, 95% CI [0.772, 0.796]
quantity                                 max R-hat   min ESS
mu                                           1.002      2936
eta_glm                                      1.000      4979
f_nn                                         1.000      9524
shape                                        0.999     13723
cann.linear_pred.layer.weight                1.002      2483
cann.linear_pred.layer.bias                  1.000      3973
loglik (eval subset)                         0.999     10330
z (random coordinates)                       1.001      4938
|z|^2                                        1.000      2758


Making predictions

In [38]:
neutra_stats = neutra_posterior_stats(gamma_bcann, neutra_samples, X_test, batch = 500)

In [40]:
print(neutra_deviance_from_stats(model = gamma_bcann, stats = neutra_stats, y = y_test, aux=w_test))

1.8302847146987915


# Horseshoe NUTS diagnostics

Start a Horseshoe GammaBCANN model

In [13]:
network_architecture = [20,15, 10]
horseshoe_model = HorseshoeGammaBCANN(
    in_features= 48,
    neural_net_params={"hidden_activation_nodes": network_architecture,
                        "hidden_activations": nn.Tanh}
).to(device)

Load HorseShoe NUTS Samples from disks and run diagnostics

In [14]:
horseshoe_nuts_run = torch.load("hmc_results/gamma_horseshoe_nuts_full_set.pt", map_location="cpu")

horseshoe_nuts_samples, horseshoe_nuts_info = horseshoe_nuts_run["samples"], horseshoe_nuts_run["info"]

In [15]:
nuts_rows = gamma_nuts_diagnostics(
    model=horseshoe_model, samples=horseshoe_nuts_samples, info=horseshoe_nuts_info,
    X_eval=X_learn, weights_eval=w_learn, y_eval=y_learn,
    n_points=500, seed=seed, device=device,
)

Device used             : cuda:2
Wall time per chain (s) : [16251.4, 16250.4, 16324.4, 16238.7, 16236.2, 16041.8]
Divergences per chain   : [19, 8, 21, 1, 23, 28]
Adapted step size       : [0.00258, 0.00136, 0.00244, 0.00147, 0.00207, 0.00273]
Gamma shape (posterior) : mean 1.364, 95% CI [1.291, 1.428]
quantity                                 max R-hat   min ESS
mu                                          10.183         3
eta_glm                                      4.643         3
f_nn                                         5.219         3
shape                                        3.472         3
cann.linear_pred.layer.bias                  4.221         3
cann.linear_pred.layer.weight                3.272         3
loglik (eval subset)                         2.977         4


# Horseshoe NeuTRa diagnostics

Load Horseshoe NeuTra samples from disk and run diagnostics

In [ ]:
horseshoe_neutra_run = torch.load("hmc_results/gamma_horseshoe_neutra_full_set.pt", map_location="cpu")

horseshoe_neutra_samples, horse_shoe_neutra_info = horseshoe_neutra_run["samples"], horseshoe_neutra_run["info"]

In [17]:
neutra_rows = neutra_diagnostics(
    model=horseshoe_model, samples=horseshoe_neutra_samples, info=horse_shoe_neutra_info,
    X_eval=X_learn, aux_eval= w_learn, y_eval=y_learn,
    n_points=500, seed=seed, device=device,
)

Device used             : ['cuda:3', 'cuda:3', 'cuda:3', 'cuda:3', 'cuda:3', 'cuda:3']
Wall time per chain (s) : [13515.7, 13890.0, 13950.7, 13875.6, 14040.1, 13913.3]
Divergences per chain   : [181, 17, 0, 0, 0, 0]
Adapted step size       : [0.00168, 0.00049, 0.00073, 0.00094, 0.00089, 0.00043]
Gamma shape (posterior) : mean 0.935, 95% CI [0.775, 1.157]
quantity                                 max R-hat   min ESS
mu                                          22.694         3
eta_glm                                     13.331         3
f_nn                                        22.819         3
shape                                       20.483         3
cann.linear_pred.layer.weight               11.666         3
cann.linear_pred.layer.bias                  2.703         4
loglik (eval subset)                         6.458         3
z (random coordinates)                      13.654         3
|z|^2                                       30.124         3
